In [ ]:
import os
import json
import gc

import matplotlib.pyplot as plt
import matplotlib

try:
    this_file = os.path.dirname(__file__)
except NameError:
    this_file = os.getcwd()

from tools.data_loader import import_database
from tools.accessions_db import AccessionDatabase
from tools.accession import Accession

export_path = '../exports/dataset/v2'
figures_path = '../exports/figures'

# matplotlib's interactive backend has a memory leak, 
# use this for rendering
plt.ioff()
matplotlib.use('Agg')

# INPUT

In [2]:
# point BASE_DIR to a directory that contains a directory called 'mg-data'
# All subdirectories under mg-data is used for analysis
BASE_DIR = os.path.join(os.path.dirname(this_file),'Database','mg-data')
ANALYSIS_DICT = {'gain':{'do_clamp':False},
                 'velocity':{'value_smoothing':3},
                 'speed':{'value_smoothing':3},
                 'acceleration':{'value_smoothing':3}}

AccDatabase = AccessionDatabase(import_database(BASE_DIR, ('Healthy control','Definite MG')))

In [ ]:
def save_json(acc:Accession, json_path:str, force_overwrite:bool=False):
    json_info = {
        'patient_name':acc.patient_name,
        'pathology_class':acc.pathology_class,
        'visit_date':acc.visit_date,
        'axis':acc.axis,
        'frequency':acc.frequency}\
            |{k:getattr(acc,k) for k in 
        ['standard_gain_dict','standard_velocity_dict','standard_speed_dict',
         'standard_acceleration_dict','standard_analysis',
         'standard_energy_dict','standard_power_dict']
    }

    json_file_path = os.path.join(json_path,f'{acc.accession_str}.json')

    if not force_overwrite and os.path.isfile(json_file_path):
        del json_info
        return
   
    with open(json_file_path,'w') as json_file:
        json.dump(json_info, json_file)

    del json_info

def save_figures(acc:Accession, figure_path:str, force_overwrite:bool=False):
    end_time = int(acc.standard_time_list[-1])
    cycle_dur = acc.standard_cycle_duration

    for start in range(0,end_time,cycle_dur):
        end = start + cycle_dur
        print(f'\t\t{start} - {end} / {end_time} begin')

        figure_name = f'{start:02d}-{end:02d} sec'
        check_path = os.path.join(figure_path,f'{figure_name}.jpg')

        if os.path.isfile(check_path):
            if not force_overwrite:
                print('\t\t\tskipped')
                continue
            os.remove(check_path)

        acc.draw(attributes_to_draw=['position','gain','speed','velocity','acceleration'],
                 region=(start,cycle_dur),
                 tick_dist=0.25,
                 use_standard=True,
                 save_name=figure_name,
                 save_dir=figure_path,
                 do_draw=False)
        print(f'\t\t\t{start} - {end} / {end_time} ended')
    
    # ENTIRE RECORDING FIGURE
    figure_name = '0000_full'
    full_fig_path = os.path.join(figure_path,f'{figure_name}.jpg')

    if os.path.isfile(full_fig_path):
        if not force_overwrite:
            return
        os.remove(full_fig_path)
        
    acc.draw(['position','gain','speed','velocity','acceleration'], tick_dist=10,
                figsize=(35,15), use_standard=True, save_dir=figure_path,save_name=figure_name,do_draw=False)

In [ ]:
accessions = list(AccDatabase.accessions.values())
del AccDatabase

while accessions:
    print(f'{len(accessions)} left')

    acc = accessions.pop()
    acc:Accession = acc.analyse(ANALYSIS_DICT).standardise_cycles()

    print('\tanalysis and standardisation finished; beginning save_path generation')

    cycle_dur = acc.standard_cycle_duration

    pathology_path = acc.pathology_class
    patient_path = acc.accession_str[:5]
    acc_path = acc.accession_str[7:]
    figure_path = figures_path
    json_path = export_path

    for path in (pathology_path,patient_path):
        json_path = os.path.join(json_path, path)
        try:
            os.mkdir(json_path)
        except:
            pass
    
    for path in (pathology_path,patient_path,acc_path):
        figure_path = os.path.join(figure_path, path)
        try:
            os.mkdir(figure_path)
        except:
            pass
    print('\tfigure_path and json_path generated; beginning json saving')
    save_json(acc, json_path, False)

    print('\tjson saved, beginning figure saving')
    save_figures(acc,figure_path,False)
    
    del acc
    gc.collect()

    print(f'this finished \n')